In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# פותר האופטימיזציה: פונקציית Qiskit מאת Q-CTRL Fire Opal
> **Note:** פונקציות Qiskit הן תכונה ניסיונית הזמינה רק למשתמשי IBM Quantum&reg; Premium Plan, Flex Plan, ו-On-Prem (דרך IBM Quantum Platform API). הן בסטטוס הפצה בתצוגה מקדימה ועשויות להשתנות.

## סקירה כללית
עם Fire Opal Optimization Solver, תוכל לפתור בעיות אופטימיזציה בקנה-מידה שימושי על חומרה קוונטית מבלי שתידרש מומחיות קוונטית. פשוט הזן את הגדרת הבעיה ברמה גבוהה, והפותר ידאג לכל השאר. כל תהליך העבודה מודע לרעש ומנצל את [ניהול הביצועים של Fire Opal](/guides/q-ctrl-performance-management) מאחורי הקלעים. הפותר מספק באופן עקבי פתרונות מדויקים לבעיות מאתגרות קלאסית, אפילו בקנה-מידה מלא של המכשיר על ה-QPU הגדולים ביותר של IBM&reg;.

הפותר גמיש וניתן להשתמש בו לפתרון בעיות אופטימיזציה קומבינטורית המוגדרות כפונקציות מטרה או כגרפים שרירותיים. אין צורך למפות בעיות לטופולוגיה של המכשיר. ניתן לפתור בעיות גם ללא אילוצים וגם עם אילוצים, כל עוד ניתן לנסח את האילוצים כמונחי עונשין. הדוגמאות הכלולות במדריך זה מדגימות כיצד לפתור בעיית אופטימיזציה ללא אילוצים ועם אילוצים בקנה-מידה שימושי, תוך שימוש בסוגי קלט שונים של הפותר. הדוגמה הראשונה כוללת בעיית Max-Cut המוגדרת על גרף בעל 156 צמתים ו-3-Regular, בעוד הדוגמה השנייה מתמודדת עם בעיית Minimum Vertex Cover בעלת 50 צמתים המוגדרת על ידי פונקציית עלות.

כדי לקבל גישה ל-Optimization Solver, [צור קשר עם Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## תיאור הפונקציה
הפותר מבצע אופטימיזציה מלאה ואוטומציה של כל האלגוריתם, החל מדיכוי שגיאות ברמת החומרה ועד מיפוי בעיות יעיל ואופטימיזציה קלאסית בלולאה סגורה. מאחורי הקלעים, הצינור של הפותר מצמצם שגיאות בכל שלב, ומאפשר את הביצועים המשופרים הנדרשים לסקאלה משמעותית. תהליך העבודה הבסיסי מבוסס על Quantum Approximate Optimization Algorithm (QAOA), שהוא אלגוריתם היברידי קוונטי-קלאסי. לסיכום מפורט של תהליך העבודה המלא של Optimization Solver, עיין ב[מאמר המפורסם](https://arxiv.org/abs/2406.01743).

![ויזואליזציה של תהליך העבודה של Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

כדי לפתור בעיה גנרית עם Optimization Solver:
1. הגדר את הבעיה שלך כפונקציית מטרה, גרף, או שרשרת ספין `SparsePauliOp`.
2. התחבר לפונקציה דרך Qiskit Functions Catalog.
3. הרץ את הבעיה עם הפותר וקבל תוצאות.
## קלטים ופלטים
### קלטים
| שם    | סוג                    | תיאור                                                                                                                                                                                         | חובה | ברירת מחדל | דוגמה |
| ---------  |-------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------| -------- | ---------- | ---------- |
| problem  | `str` or `SparsePauliOp` | אחד מהייצוגים המפורטים תחת "פורמטי בעיה מקובלים"                                                                                                                                                  | כן | N/A   |`Poly(2.0*n[0]*n[1] + 2.0*n[0]*n[2] - 3.13520241298341*n[0] + 2.0*n[1]*n[2] - 3.14469748506794*n[1] - 3.18897660121566*n[2] + 6.0, n[0], n[1], n[2], domain='RR')` |
| problem_type  | `str`                   | שם מחלקת הבעיה; בשימוש רק להגדרות בעיית גרף ושרשרת ספין, המוגבלות ל-"maxcut" או "spin_chain"; לא נדרש להגדרות בעיית פונקציית מטרה שרירותית | לא      | `None`| `"maxcut"`      |
| backend_name  | `str`                   | שם ה-Backend לשימוש                                                                                                                                                                          | לא  | ה-Backend הפחות עמוס שיש לך גישה אליו    | `"ibm_fez"`      |
| options  | `dict`                  | אפשרויות קלט, כולל הבאות: (אופציונלי) `session_id`: `str`; התנהגות ברירת המחדל יוצרת Session חדש                                                                                      | לא | `None`    | `{"session_id": "cw2q70c79ws0008z6g4g"}`     |

**פורמטי בעיה מקובלים:**
- ייצוג ביטוי פולינומיאלי של פונקציית מטרה. רצוי ליצור בפייתון עם אובייקט SymPy Poly קיים ולעצב לסטרינג באמצעות [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- ייצוג גרף של סוג בעיה ספציפי. יש ליצור את הגרף באמצעות ספריית networkx בפייתון, ולאחר מכן להמיר לסטרינג באמצעות פונקציית networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- ייצוג שרשרת ספין של בעיה ספציפית. שרשרת הספין צריכה להיות מיוצגת כאובייקט `SparsePauliOp`; ראה את [התיעוד](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) לפרטים נוספים.

**Backends נתמכים:**
הרץ את הקוד הבא כדי לראות את רשימת ה-Backends הנתמכים כרגע. אם המכשיר שלך אינו מפורט, [פנה ל-Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) להוספת תמיכה.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_marrakesh')>]

**Options:**
| Name   | Type   | Description  | Default |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | An existing Qiskit Runtime session ID  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | A list of job tags | `[]`|

### Outputs
| Name    | Type | Description | Example |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solution and metadata listed under "Result dictionary contents"         | `{'solution_bitstring_cost': 3.0, ‘final_bitstring_distribution’: {‘000001’: 100, ‘000011’: 2}, ‘iteration_count’: 3, 'solution_bitstring': ‘000001’,  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |


**Result dictionary contents:**
| Name    | Type | Description |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | The best minimum cost across all iterations        |
| final_bitstring_distribution  | `CountsDict`              | The bitstring counts dictionary associated with the minimum cost        |
| solution_bitstring | `str`              | Bitstring with the best cost in the final distribution         |
| iteration_count  | `int`              | The total number of QAOA iterations performed by the Solver        |
| variables_to_bitstring_index_map  | `float`              | The mapping from the variables to the equivalent bit in the bitstring       |
| best_parameters  | `list[float]`            | The optimized beta and gamma parameters across all iterations  |
| warnings  |`list[str]`              | The warnings produced while compiling or running QAOA (defaults to None)   |

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected Max-Cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

**אפשרויות:**
| שם   | סוג   | תיאור  | ברירת מחדל |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | מזהה session קיים של Qiskit Runtime  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | רשימת תגי job | `[]`|

### פלטים
| שם    | סוג | תיאור | דוגמה |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | פתרון ומטה-נתונים המפורטים תחת "תוכן מילון התוצאה"         | `{'solution_bitstring_cost': 3.0, 'final_bitstring_distribution': {'000001': 100, '000011': 2}, 'iteration_count': 3, 'solution_bitstring': '000001',  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |

**תוכן מילון התוצאה:**
| שם    | סוג | תיאור |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | העלות המינימלית הטובה ביותר בכל האיטרציות        |
| final_bitstring_distribution  | `CountsDict`              | מילון ספירות ה-bitstring המשויך לעלות המינימלית        |
| solution_bitstring | `str`              | ה-Bitstring עם העלות הטובה ביותר בהתפלגות הסופית         |
| iteration_count  | `int`              | המספר הכולל של איטרציות QAOA שבוצעו על ידי הפותר        |
| variables_to_bitstring_index_map  | `float`              | המיפוי מהמשתנים לביט המקביל ב-bitstring       |
| best_parameters  | `list[float]`            | פרמטרי ה-beta וה-gamma המאורגנים בכל האיטרציות  |
| warnings  |`list[str]`              | האזהרות שנוצרו בזמן קומפילציה או הרצת QAOA (ברירת מחדל None)   |
## ביצועים השוואתיים
[תוצאות ביצועים השוואתיים מפורסמות](https://arxiv.org/abs/2406.01743) מראות שהפותר פותר בהצלחה בעיות עם יותר מ-120 Qubits, ואף עולה על תוצאות שפורסמו בעבר על מכשירי quantum annealing ומלכודות יונים. מדדי הביצועים ההשוואתיים הבאים מספקים אינדיקציה גסה לדיוק וסקאלה של סוגי בעיות בהתבסס על מספר דוגמאות. המדדים בפועל עשויים להיות שונים בהתבסס על תכונות בעיה שונות, כגון מספר המונחים בפונקציית המטרה (צפיפות) והמקומיות שלהם, מספר המשתנים, וסדר הפולינום.

ה"מספר של Qubits" המצוין אינו מגבלה קשיחה אלא מייצג סף גס שבו תוכל לצפות לדיוק פתרון עקבי מאוד. גדלי בעיות גדולים יותר נפתרו בהצלחה, וניסוי מעבר למגבלות אלו מומלץ.

קישוריות Qubit שרירותית נתמכת בכל סוגי הבעיות.

| סוג בעיה    | מספר Qubits | דוגמה | דיוק | זמן כולל (שניות) | שימוש בזמן ריצה (שניות) | מספר איטרציות
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| בעיות ריבועיות עם קישוריות דלילה  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| אופטימיזציה בינארית מסדר גבוה | 156 | מודל Ising spin-glass | 100%      | 1461     | 272          | 16 |
| בעיות ריבועיות עם קישוריות צפופה | 50 | Max-Cut עם קישוריות מלאה| 100%      |  1758    | 268  | 12 |
| בעיה עם אילוצים ומונחי עונשין | 50 | Weighted Minimum Vertex Cover עם צפיפות קשתות 8% | 100%      | 1074     | 215 | 10 |
## תחילת העבודה
ראשית, אמת את עצמך באמצעות [מפתח ה-API של IBM Quantum](http://quantum.cloud.ibm.com/). לאחר מכן, בחר את פונקציית Qiskit כדלקמן. (קטע זה מניח שכבר [שמרת את חשבונך](/guides/functions#install-qiskit-functions-catalog-client) בסביבה המקומית שלך.)

In [2]:
# %pip install networkx numpy

## דוגמה: אופטימיזציה ללא אילוצים
הרץ את בעיית [החיתוך המקסימלי](https://en.wikipedia.org/wiki/Maximum_cut) (Max-Cut). הדוגמה הבאה מדגימה את יכולות הפותר על בעיית Max-Cut של גרף לא-ממושקל 3-regular בעל 156 צמתים, אך תוכל גם לפתור בעיות גרף ממושקל.
בנוסף ל-`qiskit-ibm-catalog`, תשתמש גם בחבילות הבאות להרצת דוגמה זו: `networkx` ו-`numpy`. תוכל להתקין חבילות אלו על ידי הסרת הסימון מהתא הבא אם אתה מריץ דוגמה זו ב-notebook באמצעות ה-kernel של IPython.

In [2]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [3]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

The Solver accepts a string as the problem definition input.

In [4]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

![פלט תא הקוד הקודם](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif)

הפותר מקבל סטרינג כקלט להגדרת הבעיה.

In [9]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [14]:
# Get job status
print(maxcut_job.status())

QUEUED


בדוק את [הסטטוס](/guides/functions#check-job-status) של עומס העבודה של פונקציית Qiskit שלך או קבל [תוצאות](/guides/functions#retrieve-results) כדלקמן:

In [15]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 209.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior Max-Cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [ ]:
# %pip install numpy networkx sympy

### 3. קבל את התוצאה
קבל את ערך החיתוך האופטימלי ממילון התוצאות.

> **Note:** המיפוי של המשתנים ל-bitstring עשוי להשתנות. מילון הפלט מכיל תת-מילון `variables_to_bitstring_index_map`, אשר עוזר לאמת את הסדר.

In [ ]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

A standard optimization model for weighted MVC can be formulated as follows. First, a penalty must be added for any case where an edge is not connected to a vertex in the subset. Therefore, let $n_i = 1$ if vertex $i$ is in the cover (i.e., in the subset) and $n_i = 0$ otherwise. Second, the goal is to minimize the total number of vertices in the subset, which can be represented by the following function:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

תוכל לאמת את דיוק התוצאה על ידי פתרון הבעיה קלאסית עם פותרים בקוד פתוח כמו [PuLP](https://coin-or.github.io/pulp/) אם הגרף אינו מחובר בצפיפות. בעיות עם צפיפות גבוהה עשויות לדרוש פותרים קלאסיים מתקדמים כדי לאמת את הפתרון.
## דוגמה: אופטימיזציה עם אילוצים
דוגמת Max-Cut הקודמת היא בעיית אופטימיזציה בינארית ריבועית ללא אילוצים נפוצה. Optimization Solver של Q-CTRL יכול לשמש לסוגי בעיות שונים, כולל אופטימיזציה עם אילוצים. תוכל לפתור סוגי בעיות שרירותיים על ידי הזנת הגדרת הבעיה המיוצגת כפולינום שבו אילוצים מדוגמים כמונחי עונשין.

הדוגמה הבאה מדגימה כיצד לבנות פונקציית עלות עבור בעיית אופטימיזציה עם אילוצים, [כיסוי קודקוד מינימלי](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
בנוסף לחבילות `qiskit-ibm-catalog` ו-`qiskit`, תשתמש גם בחבילות הבאות להרצת דוגמה זו: `numpy`, `networkx`, ו-`sympy`. תוכל להתקין חבילות אלו על ידי הסרת הסימון מהתא הבא אם אתה מריץ דוגמה זו ב-notebook באמצעות ה-kernel של IPython.

In [ ]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

### 1. הגדר את הבעיה
הגדר בעיית MVC אקראית על ידי יצירת גרף עם צמתים בעלי משקל אקראי.

In [ ]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

![פלט תא הקוד הקודם](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif)

ניתן לנסח מודל אופטימיזציה סטנדרטי ל-MVC ממושקל כדלקמן. ראשית, יש להוסיף עונש עבור כל מקרה שבו קשת אינה מחוברת לקודקוד בתת-הקבוצה. לכן, נגדיר $n_i = 1$ אם קודקוד $i$ נמצא בכיסוי (כלומר, בתת-הקבוצה) ו-$n_i = 0$ אחרת. שנית, המטרה היא למזער את המספר הכולל של הקודקודים בתת-הקבוצה, אשר ניתן לייצג באמצעות הפונקציה הבאה:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
print(mvc_job.status())

כעת כל קשת בגרף צריכה לכלול לפחות נקודת קצה אחת מהכיסוי, אשר ניתן לבטא כאי-שוויון:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

כל מקרה שבו קשת אינה מחוברת לקודקוד הכיסוי חייב לקבל עונש. ניתן לייצג זאת בפונקציית העלות על ידי הוספת עונש בצורה $P(1-n_i-n_j+n_i n_j)$ כאשר $P$ הוא קבוע עונשין חיובי. לפיכך, חלופה ללא אילוצים לאי-השוויון עם אילוצים ל-MVC ממושקל היא:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [ ]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


### 2. הרץ את הבעיה